# 🤖 Intellix — Treinamento do Modelo de NLU**Leia isto antes de rodar qualquer coisa.**Este notebook treina o modelo de IA do Intellix: aquele que recebe *"quanto faturei em março"* e devolve `intenção = total_vendas_periodo` e `entidades = {periodo: março}`.### Como usarAperte ▶ em **cada célula, de cima para baixo, sem pular**. Se uma der erro, conserte antes de seguir.### O que você precisa ter em mãos- `dados/dataset.jsonl` — gerado pelo `gerar_dataset.py`- `dados/teste_manual.jsonl` — as frases que **vocês** escreveram à mão### Antes de começar`Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU T4** → Salvar.⚠️ **O Colab apaga tudo quando desconecta.** A célula 2 conecta o Google Drive pra salvar o modelo lá. Não pule.

## 1. Preparar o ambiente

In [ ]:
import torchprint("PyTorch:", torch.__version__)if torch.cuda.is_available():    dispositivo = torch.device("cuda")    print("✅ GPU ligada:", torch.cuda.get_device_name(0))else:    dispositivo = torch.device("cpu")    print("⚠️  Sem GPU. Vai funcionar, só mais devagar.")    print("   Ambiente de execução → Alterar tipo → GPU T4")# seqeval calcula o F1 das entidades do jeito certo (por entidade, não por token)!pip install -q seqevalprint("✅ pronto")

In [ ]:
# Conecta o Google Drive para salvar o modelo.# Vai abrir uma janela pedindo permissão da sua conta Google. Aceite.from google.colab import drivefrom pathlib import Pathdrive.mount('/content/drive')PASTA = Path('/content/drive/MyDrive/intellix')PASTA.mkdir(parents=True, exist_ok=True)print("✅ Modelos serão salvos em:", PASTA)

## 2. Subir os dadosVai aparecer um botão **"Escolher arquivos"**. Selecione os **dois** arquivos de uma vez:`dataset.jsonl` e `teste_manual.jsonl`.

In [ ]:
from google.colab import filesimport json, ossubidos = files.upload()def carregar(nome):    with open(nome, encoding="utf-8") as f:        return [json.loads(l) for l in f if l.strip()]dados_sinteticos = carregar("dataset.jsonl")dados_manuais    = carregar("teste_manual.jsonl")print(f"\n✅ {len(dados_sinteticos)} exemplos gerados por molde")print(f"✅ {len(dados_manuais)} exemplos escritos à mão (conjunto de teste real)")# Trava de segurança: nunca treinar com dataset quebradofor nome, dados in [("dataset", dados_sinteticos), ("teste_manual", dados_manuais)]:    ruins = [d for d in dados if len(d["tokens"]) != len(d["tags"])]    assert not ruins, f"❌ {nome}: {len(ruins)} exemplos com tokens/tags desalinhados!"print("✅ tokens e tags alinhados em todos os exemplos")

## 3. Separar treino / validação / testeTrês montes de dados, com três funções bem diferentes:| Monte | De onde vem | Serve pra ||---|---|---|| **Treino** (70%) | moldes | o modelo aprender || **Validação** (15%) | moldes | decidir quando parar de treinar || **Teste sintético** (15%) | moldes | comparar com o teste real || **Teste real** | escrito à mão | 🎯 **o número que vai pro TCC** |Por que dois testes? Porque a **diferença entre eles** é a coisa mais interessante do TCC de vocês. Se der 97% no sintético e 89% no real, esses 8 pontos são a medida de quanto o modelo decorou os moldes. Isso é um achado científico, não um fracasso.

In [ ]:
from sklearn.model_selection import train_test_split# stratify = garante a mesma proporção de cada intenção nos três montesrotulos = [d["intencao"] for d in dados_sinteticos]treino, resto = train_test_split(    dados_sinteticos, test_size=0.30, random_state=42, stratify=rotulos)resto_rotulos = [d["intencao"] for d in resto]validacao, teste_sintetico = train_test_split(    resto, test_size=0.50, random_state=42, stratify=resto_rotulos)teste_real = dados_manuaisprint(f"Treino:           {len(treino):>5}")print(f"Validação:        {len(validacao):>5}")print(f"Teste sintético:  {len(teste_sintetico):>5}")print(f"Teste real:       {len(teste_real):>5}  🎯")

## 4. Construir o vocabulárioO modelo não lê letras, só números. Então cada palavra vira um número.Dois números especiais:- **`<PAD>` = 0** → enchimento. Frases têm tamanhos diferentes, mas o modelo precisa de blocos iguais.- **`<UNK>` = 1** → palavra que ele nunca viu no treino.🚨 **O vocabulário é tão importante quanto o modelo.** Sem ele, os pesos treinados não significam nada. Por isso ele vai ser salvo *dentro* do mesmo arquivo do modelo, lá no fim.⚠️ O vocabulário sai **só do treino**. Se usar validação ou teste aqui, você está deixando o modelo espiar a prova.

In [ ]:
from collections import Countercontagem = Counter(tok for ex in treino for tok in ex["tokens"])vocab = {"<PAD>": 0, "<UNK>": 1}for palavra, n in contagem.most_common():    if n >= 2:                      # palavra que aparece 1x só vira <UNK>        vocab[palavra] = len(vocab)INTENCOES = sorted({d["intencao"] for d in dados_sinteticos})intent2id = {nome: i for i, nome in enumerate(INTENCOES)}TIPOS = ["PERIODO","DATA","PRODUTO","CATEGORIA","CLIENTE","VENDEDOR","CANAL","PAGAMENTO","NUMERO"]tag2id = {"O": 0}for t in TIPOS:    tag2id[f"B-{t}"] = len(tag2id)    tag2id[f"I-{t}"] = len(tag2id)id2tag = {i: t for t, i in tag2id.items()}id2intent = {i: n for n, i in intent2id.items()}print(f"Vocabulário: {len(vocab)} palavras")print(f"Intenções:   {len(intent2id)}")print(f"Tags:        {len(tag2id)}")assert len(intent2id) == 12 and len(tag2id) == 19# quantas palavras do teste real o modelo nunca viu?palavras_teste = {tok for ex in teste_real for tok in ex["tokens"]}desconhecidas = palavras_teste - set(vocab)print(f"\nPalavras do teste real fora do vocabulário: {len(desconhecidas)}/{len(palavras_teste)}")if desconhecidas:    print("Exemplos:", sorted(desconhecidas)[:15])    print("👉 Cada uma dessas vira <UNK>. Se forem muitas, faltam moldes/valores.")

## 5. Preparar os lotes (padding)O modelo processa várias frases de uma vez — um **lote** (*batch*). Mas as frases têm tamanhos diferentes, e a GPU só trabalha com blocos retangulares. Então completamos as menores com `<PAD>`:```"faturamento março"        -> [45, 12,  0,  0,  0]"quanto vendi em março ?"  -> [ 8, 33, 15, 12, 27]```⚠️ **Detalhe que quase todo mundo erra:** as tags são completadas com **-100**, não com 0. Porque `0` é a tag `"O"` — se usássemos 0, o modelo gastaria esforço aprendendo a prever `"O"` em cima de enchimento que não existe. O `-100` é o valor que o PyTorch entende como *"ignora isto aqui"*.

In [ ]:
from torch.utils.data import Dataset, DataLoaderimport torchIGNORAR = -100class DatasetNLU(Dataset):    def __init__(self, exemplos):        self.exemplos = exemplos    def __len__(self):        return len(self.exemplos)    def __getitem__(self, i):        ex = self.exemplos[i]        ids_tokens = [vocab.get(t, vocab["<UNK>"]) for t in ex["tokens"]]        ids_tags   = [tag2id[t] for t in ex["tags"]]        return ids_tokens, ids_tags, intent2id[ex["intencao"]]def juntar_lote(lote):    tokens, tags, intencoes = zip(*lote)    maior = max(len(t) for t in tokens)    x  = [t + [0] * (maior - len(t))         for t in tokens]   # PAD = 0    yt = [t + [IGNORAR] * (maior - len(t))   for t in tags]     # PAD = -100    return (torch.tensor(x), torch.tensor(yt), torch.tensor(intencoes))def criar_loader(dados, embaralhar):    return DataLoader(DatasetNLU(dados), batch_size=32,                      shuffle=embaralhar, collate_fn=juntar_lote)loader_treino    = criar_loader(treino, True)loader_validacao = criar_loader(validacao, False)loader_t_sint    = criar_loader(teste_sintetico, False)loader_t_real    = criar_loader(teste_real, False)x, yt, yi = next(iter(loader_treino))print("Formato de um lote:")print(f"  tokens:   {tuple(x.shape)}   (32 frases x {x.shape[1]} palavras)")print(f"  tags:     {tuple(yt.shape)}")print(f"  intenção: {tuple(yi.shape)}")

## 6. O modeloChegamos nele. São **25 linhas**. Sério, é isso.```palavras -> [Embedding] -> [LSTM bidirecional] -> ┬-> [Cabeça 1] -> intenção                                                  └-> [Cabeça 2] -> tags BIO```**Embedding**: cada palavra vira um vetor de 100 números. No começo são aleatórios; o treino ajusta até palavras parecidas ficarem com vetores parecidos.**LSTM bidirecional**: lê a frase da esquerda pra direita *e* da direita pra esquerda. Por que os dois lados? Em *"quanto vendi de camiseta em março"*, pra entender que "camiseta" é produto ajuda saber que vem "em março" depois. Lendo só pra frente, o modelo ainda não sabe disso.**As duas cabeças** dividem o mesmo miolo — é o **aprendizado multitarefa** da seção 6.6.2 do documento de vocês. As tarefas se ajudam: quem entendeu que a frase pergunta faturamento entende melhor que "março" é período.**Dropout**: desliga 30% dos neurônios ao acaso durante o treino. Parece sabotagem, mas força o modelo a não depender de um detalhe só. É uma das defesas contra overfitting da seção 4.5.

In [ ]:
import torch.nn as nnclass ModeloNLU(nn.Module):    def __init__(self, tam_vocab, n_intencoes=12, n_tags=19,                 dim_emb=100, dim_oculta=128, dropout=0.3):        super().__init__()        self.emb = nn.Embedding(tam_vocab, dim_emb, padding_idx=0)        self.lstm = nn.LSTM(dim_emb, dim_oculta, batch_first=True,                            bidirectional=True)        self.dropout = nn.Dropout(dropout)        self.cabeca_intencao = nn.Linear(dim_oculta * 2, n_intencoes)        self.cabeca_tags     = nn.Linear(dim_oculta * 2, n_tags)    def forward(self, x):        mascara = (x != 0).unsqueeze(-1).float()      # onde NÃO é enchimento        e = self.dropout(self.emb(x))                 # (lote, palavras, 100)        saidas, _ = self.lstm(e)                      # (lote, palavras, 256)        # Representação da frase inteira = média das palavras REAIS.        # A máscara garante que o enchimento não entre na média — se entrasse,        # frases curtas ficariam com a representação "diluída" por zeros.        frase = (saidas * mascara).sum(1) / mascara.sum(1).clamp(min=1)        frase = self.dropout(frase)        return self.cabeca_intencao(frase), self.cabeca_tags(saidas)modelo = ModeloNLU(len(vocab)).to(dispositivo)n_params = sum(p.numel() for p in modelo.parameters())print(modelo)print(f"\nParâmetros: {n_params:,}")print(f"Tamanho estimado: {n_params * 4 / 1024**2:.1f} MB")print("\n👉 Compare com o GPT-4 (~1.7 trilhão de parâmetros).")print("   Esse tamanho é o que faz caber nos 512MB de RAM do Render (seção 6.8).")

## 7. TreinarO loop é sempre o mesmo:> mostra frases → o modelo chuta → compara com o certo → mede o erro → ajusta os pesos → repete**Erro total = α · erro_intenção + (1−α) · erro_entidades** — exatamente a fórmula da seção 6.6.5 do documento.**Early stopping**: a acurácia de validação sobe, sobe, e uma hora começa a **cair** enquanto a de treino continua subindo. Isso é o overfitting acontecendo ao vivo. Guardamos o modelo da melhor época, não o da última.

In [ ]:
import copyALPHA = 0.5           # peso entre as duas tarefas (teste 0.3 e 0.7 depois)EPOCAS = 40PACIENCIA = 6         # para se passar 6 épocas sem melhorarotimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)perda_int  = nn.CrossEntropyLoss()perda_tags = nn.CrossEntropyLoss(ignore_index=IGNORAR)   # ignora o enchimentodef acuracia_intencao(loader):    modelo.eval()    certos = total = 0    with torch.no_grad():        for x, yt, yi in loader:            x, yi = x.to(dispositivo), yi.to(dispositivo)            logits_i, _ = modelo(x)            certos += (logits_i.argmax(1) == yi).sum().item()            total += yi.size(0)    return certos / totalhistorico = []melhor_acc, melhor_estado, epocas_sem_melhora = 0.0, None, 0for epoca in range(1, EPOCAS + 1):    modelo.train()    soma_erro = 0.0    for x, yt, yi in loader_treino:        x, yt, yi = x.to(dispositivo), yt.to(dispositivo), yi.to(dispositivo)        otimizador.zero_grad()        logits_i, logits_t = modelo(x)        erro_i = perda_int(logits_i, yi)        erro_t = perda_tags(logits_t.reshape(-1, len(tag2id)), yt.reshape(-1))        erro = ALPHA * erro_i + (1 - ALPHA) * erro_t        erro.backward()        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 5.0)  # evita explosão        otimizador.step()        soma_erro += erro.item()    acc_treino = acuracia_intencao(loader_treino)    acc_val    = acuracia_intencao(loader_validacao)    historico.append((epoca, soma_erro / len(loader_treino), acc_treino, acc_val))    if acc_val > melhor_acc:        melhor_acc = acc_val        melhor_estado = copy.deepcopy(modelo.state_dict())        epocas_sem_melhora = 0        marca = "⭐ melhor"    else:        epocas_sem_melhora += 1        marca = ""    print(f"época {epoca:>2} | erro {soma_erro/len(loader_treino):.4f} "          f"| treino {acc_treino:.3f} | validação {acc_val:.3f} {marca}")    if epocas_sem_melhora >= PACIENCIA:        print(f"\n⏹️  Early stopping: {PACIENCIA} épocas sem melhorar.")        breakmodelo.load_state_dict(melhor_estado)   # volta pro melhorprint(f"\n✅ Melhor acurácia de validação: {melhor_acc:.3f}")

In [ ]:
# Gráfico do treino — vai direto pro TCCimport matplotlib.pyplot as pltep   = [h[0] for h in historico]erro = [h[1] for h in historico]at   = [h[2] for h in historico]av   = [h[3] for h in historico]fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))a1.plot(ep, erro, marker='o', color='crimson')a1.set_title("Erro no treino (tem que cair)")a1.set_xlabel("época"); a1.set_ylabel("erro"); a1.grid(alpha=.3)a2.plot(ep, at, marker='o', label="treino")a2.plot(ep, av, marker='s', label="validação")a2.axvline(ep[av.index(max(av))], ls='--', c='gray', label="melhor época")a2.set_title("Acurácia — se as linhas se afastarem, é overfitting")a2.set_xlabel("época"); a2.set_ylabel("acurácia"); a2.legend(); a2.grid(alpha=.3)plt.tight_layout()plt.savefig('curva_treino.png', dpi=150)plt.show()print("👉 Se a linha azul (treino) sobe e a laranja (validação) cai, o modelo está decorando.")

## 8. Avaliar — os números do TCCAgora vem a hora da verdade. Rodamos **os dois testes** e comparamos.Metas do documento (Tabela 1, seção 8.4):- Acurácia de intenção **≥ 85%**- F1 das entidades **≥ 0,80**O número que vale é o do **teste real**.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_scorefrom seqeval.metrics import classification_report as report_entidadesfrom seqeval.metrics import f1_score as f1_entidadesdef avaliar(loader, dados, nome):    modelo.eval()    y_int_real, y_int_prev = [], []    y_tag_real, y_tag_prev = [], []    confiancas = []    with torch.no_grad():        for x, yt, yi in loader:            x = x.to(dispositivo)            logits_i, logits_t = modelo(x)            probs = torch.softmax(logits_i, dim=1)            conf, pred_i = probs.max(1)            confiancas += conf.cpu().tolist()            y_int_real += yi.tolist()            y_int_prev += pred_i.cpu().tolist()            pred_t = logits_t.argmax(-1).cpu()            for linha_real, linha_prev in zip(yt, pred_t):                reais, prevs = [], []                for r, p in zip(linha_real.tolist(), linha_prev.tolist()):                    if r == IGNORAR:       # pula o enchimento                        continue                    reais.append(id2tag[r])                    prevs.append(id2tag[p])                y_tag_real.append(reais)                y_tag_prev.append(prevs)    acc = accuracy_score(y_int_real, y_int_prev)    f1e = f1_entidades(y_tag_real, y_tag_prev)    print(f"\n{'='*62}")    print(f"  {nome}  ({len(dados)} exemplos)")    print(f"{'='*62}")    print(f"  Acurácia de intenção : {acc:.3f}   {'✅' if acc >= 0.85 else '❌'} (meta 0.85)")    print(f"  F1 das entidades     : {f1e:.3f}   {'✅' if f1e >= 0.80 else '❌'} (meta 0.80)")    print(f"  Confiança média      : {sum(confiancas)/len(confiancas):.3f}")    return acc, f1e, y_int_real, y_int_prev, y_tag_real, y_tag_prevr_sint = avaliar(loader_t_sint, teste_sintetico, "TESTE SINTÉTICO (frases de molde)")r_real = avaliar(loader_t_real, teste_real, "TESTE REAL (escritas à mão) 🎯")print(f"\n{'='*62}")print(f"  A DIFERENÇA — este é o achado do TCC")print(f"{'='*62}")print(f"  Intenção: {r_sint[0]:.3f} (molde) vs {r_real[0]:.3f} (real) "      f"→ queda de {(r_sint[0]-r_real[0])*100:.1f} pontos")print(f"  Entidades: {r_sint[1]:.3f} (molde) vs {r_real[1]:.3f} (real) "      f"→ queda de {(r_sint[1]-r_real[1])*100:.1f} pontos")print("\n  Essa queda mede o quanto o modelo decorou os moldes em vez de")print("  aprender português. Quanto menor, melhor generalizou.")

In [ ]:
# Relatório por intenção — a tabela do TCCprint("INTENÇÃO — teste real, por classe:\n")print(classification_report(r_real[2], r_real[3],                            labels=list(range(len(INTENCOES))),                            target_names=INTENCOES, zero_division=0))print("\nENTIDADES — teste real, por tipo:\n")print(report_entidades(r_real[4], r_real[5], zero_division=0))

In [ ]:
# Matriz de confusão — mostra QUAIS intenções o modelo troca entre siimport seaborn as snsimport numpy as npmc = confusion_matrix(r_real[2], r_real[3], labels=list(range(len(INTENCOES))))plt.figure(figsize=(11, 9))sns.heatmap(mc, annot=True, fmt='d', cmap='Blues',            xticklabels=INTENCOES, yticklabels=INTENCOES, cbar=False)plt.xlabel("O que o modelo previu")plt.ylabel("O que era de verdade")plt.title("Matriz de confusão — teste real")plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)plt.tight_layout()plt.savefig('matriz_confusao.png', dpi=150)plt.show()# Onde ele mais erraprint("\nOs erros mais comuns:")erros = [(INTENCOES[i], INTENCOES[j], mc[i][j])         for i in range(len(INTENCOES)) for j in range(len(INTENCOES))         if i != j and mc[i][j] > 0]for real, prev, n in sorted(erros, key=lambda x: -x[2])[:8]:    print(f"  {n:>2}x  '{real}'  virou  '{prev}'")print("\n👉 Conserto: escreva mais moldes que deixem a diferença clara entre elas.")print("   Não mexa no modelo. O problema quase sempre está no dataset.")

## 9. Medir a velocidadeO documento promete **resposta em menos de 3 segundos** (Tabela 1). Quase todo esse tempo vai ser banco de dados e rede — a inferência do modelo é a parte mais rápida. Vamos provar isso com número.

In [ ]:
import timemodelo.eval()modelo_cpu = ModeloNLU(len(vocab))       # CPU, porque é assim que roda no Rendermodelo_cpu.load_state_dict(modelo.state_dict())modelo_cpu.eval()frase_teste = torch.tensor([[vocab.get(t, 1) for t in                             ["quanto", "faturei", "em", "março"]]])with torch.no_grad():    for _ in range(10):                  # aquece        modelo_cpu(frase_teste)    inicio = time.perf_counter()    for _ in range(200):        modelo_cpu(frase_teste)    tempo = (time.perf_counter() - inicio) / 200 * 1000print(f"Tempo de inferência (CPU, 1 frase): {tempo:.2f} ms")print(f"Orçamento total do Intellix: 3000 ms")print(f"O modelo usa {tempo/3000*100:.3f}% do orçamento.")print("\n👉 Guarde esse número: é uma das colunas do comparativo BiLSTM vs Transformer.")

## 10. Salvar o modelo**Tudo num arquivo só**: pesos + vocabulário + dicionários + métricas + versão.Isto é literalmente o *"manifesto versionado"* da seção 7.8 do documento. Vocês estão cumprindo o que escreveram sem perceber.🚨 Um `.pt` só com os pesos, sem o vocabulário, é um arquivo inútil. Nunca separe os dois.

In [ ]:
from datetime import datetimeVERSAO = "1.0.0"pacote = {    "state_dict": modelo.state_dict(),    "vocab": vocab,    "intent2id": intent2id,    "tag2id": tag2id,    "config": {        "arquitetura": "BiLSTM",        "tam_vocab": len(vocab),        "dim_emb": 100,        "dim_oculta": 128,        "dropout": 0.3,        "n_intencoes": len(intent2id),        "n_tags": len(tag2id),    },    "treino": {        "alpha": ALPHA,        "learning_rate": 1e-3,        "batch_size": 32,        "epocas_rodadas": len(historico),        "n_treino": len(treino),        "n_validacao": len(validacao),    },    "metricas": {        "acc_intencao_teste_real": round(r_real[0], 4),        "f1_entidades_teste_real": round(r_real[1], 4),        "acc_intencao_teste_sintetico": round(r_sint[0], 4),        "f1_entidades_teste_sintetico": round(r_sint[1], 4),        "acc_validacao": round(melhor_acc, 4),        "ms_inferencia_cpu": round(tempo, 2),    },    "versao": VERSAO,    "data": datetime.now().isoformat(timespec="seconds"),}nome = f"intellix_nlu_v{VERSAO}.pt"torch.save(pacote, PASTA / nome)          # Google Drive (permanente)torch.save(pacote, nome)                  # local (pra baixar)import osprint(f"✅ Salvo em {PASTA / nome}")print(f"   Tamanho: {os.path.getsize(nome)/1024**2:.1f} MB")print(f"   Cabe nos 512MB do Render? {'✅ sim' if os.path.getsize(nome)/1024**2 < 100 else '⚠️  apertado'}")files.download(nome)     # baixa pro seu computador

## 11. Testar na mãoO momento divertido. Escreva o que quiser e veja o modelo responder.Repare no **fallback**: se a confiança for menor que 0,70, ele se recusa a responder e pede pra reformular — exatamente o comportamento da seção 7.6 do documento. Testem frases fora do escopo ("qual a capital da França?") pra ver o fallback funcionando.

In [ ]:
import re, unicodedataLIMIAR_CONFIANCA = 0.70      # seção 7.6 do documentoABREVIACOES = {"qnt":"quanto","qto":"quanto","qtd":"quantidade","qtas":"quantas",               "qts":"quantos","vlr":"valor","tkt":"ticket","fat":"faturamento",               "vc":"você","q":"que","pra":"para","pro":"para o"}def normalizar(texto):    texto = unicodedata.normalize("NFC", texto.lower().strip())    texto = re.sub(r"\s+", " ", texto)    return " ".join(ABREVIACOES.get(p, p) for p in texto.split())def tokenizar(texto):    # ⚠️ TEM que ser idêntica à do gerar_dataset.py e à do backend    return re.findall(r"\w+|[^\w\s]", normalizar(texto), re.UNICODE)def prever(pergunta):    tokens = tokenizar(pergunta)    if not tokens:        return None    x = torch.tensor([[vocab.get(t, 1) for t in tokens]])    modelo_cpu.eval()    with torch.no_grad():        logits_i, logits_t = modelo_cpu(x)    probs = torch.softmax(logits_i, 1)[0]    conf, idx = probs.max(0)    tags = [id2tag[i] for i in logits_t.argmax(-1)[0].tolist()]    # junta os tokens B-X I-X I-X numa entidade só    entidades, atual, tipo = [], [], None    for tok, tag in zip(tokens, tags):        if tag.startswith("B-"):            if atual: entidades.append((tipo, " ".join(atual)))            atual, tipo = [tok], tag[2:]        elif tag.startswith("I-") and tipo == tag[2:]:            atual.append(tok)        else:            if atual: entidades.append((tipo, " ".join(atual)))            atual, tipo = [], None    if atual: entidades.append((tipo, " ".join(atual)))    return id2intent[idx.item()], conf.item(), entidades, probsdef responder(pergunta):    r = prever(pergunta)    print(f"\n💬 {pergunta!r}")    if not r:        print("   ❌ frase vazia"); return    intencao, conf, entidades, probs = r    if conf < LIMIAR_CONFIANCA:        print(f"   🤔 FALLBACK (confiança {conf:.2f} < {LIMIAR_CONFIANCA})")        print("   → 'Não entendi bem. Pode reformular a pergunta?'")        top3 = torch.topk(probs, 3)        print("   (palpites: " + ", ".join(            f"{id2intent[i.item()]} {p:.2f}" for p, i in zip(top3.values, top3.indices)) + ")")        return    print(f"   ✅ intenção : {intencao}  (confiança {conf:.2f})")    print(f"   📌 entidades: {entidades if entidades else 'nenhuma'}")for p in [    "quanto faturei em março",    "quais foram meus 5 produtos mais vendidos em abril",    "oi, quanto que eu vendi mês passado?",    "o pix tá vendendo mais que o cartão?",    "quem é meu melhor vendedor esse ano",    "tem alguma coisa encalhada no estoque",    "qual a capital da França?",          # fora do escopo → tem que dar fallback    "asdkjhaskdjh",                        # lixo → tem que dar fallback]:    responder(p)

In [ ]:
# 👉 Escreva a sua aqui e rode:responder("quanto vendi de camiseta branca em abril")

## 12. E agora?✅ Vocês têm um modelo treinado, medido e salvo. Isso é o núcleo do TCC.**Próximos passos, nesta ordem:**1. **Melhorar o número do teste real.** Se ficou abaixo de 85%, o conserto quase nunca é no modelo — é **mais moldes e mais valores** no `moldes.py`. Olhem a matriz de confusão pra saber quais intenções atacar.2. **Testar o α.** Rodem com `ALPHA = 0.3` e `0.7` e anotem. Vira uma tabelinha no TCC.3. **Integrar no FastAPI.** Carreguem o `.pt`, copiem as funções `tokenizar` e `prever` deste notebook. A `tokenizar` **tem que ser idêntica** — é o bug mais silencioso que existe.4. **Só então, o Transformer.** Troquem o miolo (Embedding+LSTM) por um encoder pequeno. Tudo o mais neste notebook continua igual. Guardem os 4 números pra comparar: acurácia, F1, MB, ms.**Salve este notebook** em `Arquivo → Salvar uma cópia no GitHub`.